In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

def load_nuscenes_bin(file_path):
    """Laad nuScenes pointcloud (.bin) bestand."""
    points = np.fromfile(file_path, dtype=np.float32)
    # nuScenes format: (x, y, z, intensity, ring_index)
    points = points.reshape(-1, 5)
    xyz = points[:, :3]
    return xyz

def compute_polar_grid_slopes(points, plane_model, 
                             r_step=5.0, 
                             theta_step_deg=15.0, 
                             distance_thresh=0.2, 
                             min_points=10, 
                             plane_thickness=0.1, 
                             plane_iterations=1):
    """
    Verdeelt de pointcloud in polaire cellen en berekent de helling per cel via RANSAC.
    """
    a, b, c, d = plane_model
    normal = np.array([a, b, c])
    normal = normal / np.linalg.norm(normal)

    # 1. Isoleer grondpunten (Let op: distance_thresh moet positief zijn voor np.abs)
    distances = points @ normal + d
    ground_points = points[np.abs(distances) < distance_thresh]
    
    if len(ground_points) == 0:
        print("Geen punten gevonden binnen de distance_thresh!")
        return []

    # 2. Omzetten naar Polair
    x = ground_points[:, 0]
    y = ground_points[:, 1]
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)

    # Definieer bins
    r_bins = np.arange(0, r.max() + r_step, r_step)
    theta_bins = np.deg2rad(np.arange(-180, 180 + theta_step_deg, theta_step_deg))

    z_axis = np.array([0, 0, 1])
    results = []

    # 3. Itereer door cellen
    for i in range(len(r_bins) - 1):
        for j in range(len(theta_bins) - 1):
            mask = (
                (r >= r_bins[i]) & (r < r_bins[i+1]) &
                (theta >= theta_bins[j]) & (theta < theta_bins[j+1])
            )
            cell_pts = ground_points[mask]

            if len(cell_pts) < min_points:
                continue

            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(cell_pts)

            try:
                # LOKALE RANSAC PLANE FITTING
                # Hier worden de variabelen gebruikt die je buiten de functie definieert
                local_plane, _ = pcd.segment_plane(plane_thickness, 3, plane_iterations)
                
                la, lb, lc, _ = local_plane
                l_norm = np.array([la, lb, lc])
                l_norm /= np.linalg.norm(l_norm)

                if l_norm[2] < 0: l_norm = -l_norm

                slope = np.degrees(np.arccos(np.clip(np.dot(l_norm, z_axis), -1, 1)))
                
                results.append({
                    'r_bounds': (r_bins[i], r_bins[i+1]),
                    'theta_bounds': (theta_bins[j], theta_bins[j+1]),
                    'slope': slope
                })
            except:
                continue
    return results

def visualize_polar_curvature(results):
    if not results: 
        print("Geen resultaten om te tonen.")
        return

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='polar')

    slopes = np.array([res['slope'] for res in results])
    vmax = np.percentile(slopes, 95) if len(slopes) > 0 else 15
    cmap = plt.get_cmap('RdYlGn_r')

    for res in results:
        r0, r1 = res['r_bounds']
        t0, t1 = res['theta_bounds']
        ax.bar(t0, r1-r0, width=t1-t0, bottom=r0, 
               color=cmap(res['slope']/vmax), align='edge', alpha=0.8)

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, vmax))
    plt.colorbar(sm, ax=ax, label='Helling (graden)')
    plt.title("Polar Ground Curvature Map")
    plt.show()

# --- PARAMETERS (Buiten de functie gedefinieerd) ---

FILE_PATH = r'C:\Users\Lars Wissink\OneDrive\Documenten\lars wissink\WB TU Delft\jaar 3\bep\pointcloud.bin'

R_STEP           = 2.0   
THETA_STEP       = 10.0  
DIST_THRESH      = 1.5   # Punten binnen 1.5m van het referentievlak
MIN_POINTS_CELL  = 10    
PLANE_MODEL      = (0, 0, 1, 1.8786) # Let op de haakjes (tuple)

# Jouw RANSAC specifieke parameters
PLANE_THICKNESS  = 0.1  # De 'distance_threshold' in segment_plane
PLANE_ITERATIONS = 100
    # Verhoogd voor betere resultaten (was 10)

# --- UITVOERING ---

# 1. Punten laden
points = load_nuscenes_bin(FILE_PATH)
print('Loaded points:', points.shape)

# 2. Berekening
# We geven de parameters expliciet mee aan de functie
polar_results = compute_polar_grid_slopes(
    points, 
    plane_model=PLANE_MODEL,
    r_step=R_STEP, 
    theta_step_deg=THETA_STEP,
    distance_thresh=DIST_THRESH, 
    min_points=MIN_POINTS_CELL,
    plane_thickness=PLANE_THICKNESS,    # Hier koppel je buiten-variabele aan functie-argument
    plane_iterations=PLANE_ITERATIONS   # Hier koppel je buiten-variabele aan functie-argument
)

# 3. Visualisatie
visualize_polar_curvature(polar_results)

In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
from rosbags.highlevel import AnyReader
from pathlib import Path
from dateutil import parser
import struct

# --- 1. DATA LADEN UIT MCAP ---

def get_lidar_by_timestamp(bag_path, target_time_str, topic_name=None):
    """Zoekt het dichtstbijzijnde LiDAR frame op basis van een timestamp string."""
    bag_path_obj = Path(bag_path)
    
    # Zet de tijd-string om naar nanoseconden (Unix epoch)
    dt = parser.parse(target_time_str)
    target_ns = int(dt.timestamp() * 1e9)
    print(f"Zoeken naar tijdstip (NS): {target_ns}")

    with AnyReader([bag_path_obj]) as reader:
        # Automatisch topic selecteren als er geen is opgegeven
        if topic_name is None:
            connections = [x for x in reader.connections if 'PointCloud2' in x.msgtype]
            if not connections: 
                print("Geen PointCloud2 topics gevonden!")
                return None, None
            connection = connections[0]
            print(f"Geselecteerd topic: {connection.topic}")
        else:
            connections = [x for x in reader.connections if x.topic == topic_name]
            if not connections: 
                print(f"Topic {topic_name} niet gevonden!")
                return None, None
            connection = connections[0]

        try:
            # We gebruiken 'start' om direct naar de juiste tijd te springen
            for conn, timestamp, rawdata in reader.messages(connections=[connection], start=target_ns):
                msg = reader.deserialize(rawdata, conn.msgtype)
                
                # Punten extraheren uit bytes
                point_step = msg.point_step
                num_floats_per_point = point_step // 4
                raw_floats = np.frombuffer(msg.data, dtype=np.float32)
                
                # Voorkom reshape errors door alleen volledige punten te pakken
                total_complete_points = len(raw_floats) // num_floats_per_point
                clean_floats = raw_floats[:total_complete_points * num_floats_per_point]
                points = clean_floats.reshape(-1, num_floats_per_point)
                xyz = points[:, :3]

                # --- OPSCHONING (Cruciaal voor jouw foutmelding) ---
                # 1. Verwijder NaN en Inf waarden
                xyz = xyz[np.all(np.isfinite(xyz), axis=1)]
                
                # 2. Verwijder extreme uitschieters (>200m) om geheugenfouten te voorkomen
                dist_sq = np.sum(xyz**2, axis=1)
                xyz = xyz[dist_sq < 15**2]

                if len(xyz) == 0:
                    print("Waarschuwing: Geen geldige punten overgebleven na filtering.")
                    continue

                print(f"Gevonden frame op bag-tijd: {timestamp}")
                print(f"Aantal punten in frame: {len(xyz)}")
                return xyz, timestamp
        except Exception as e:
            print(f"Fout tijdens het lezen: {e}")
            
    return None, None

# --- 2. ANALYSE FUNCTIE ---



# Paden en Tijd
MCAP_FILE_PATH = r"C:\rosbag_0.mcap"
TARGET_TIME    = "2025-11-12 4:22:58.461 PM +01"

# Parameters
PLANE_MODEL      = (0, 0, 1, 1) # (a, b, c, d)
R_STEP           = 1.0
THETA_STEP       = 10.0
DIST_THRESH      = 1.0 
PLANE_THICKNESS  = 0.05
PLANE_ITERATIONS = 50

# START
print("--- Start Proces ---")
points, found_ts = get_lidar_by_timestamp(MCAP_FILE_PATH, TARGET_TIME)

if points is not None:
    print("Berekenen van hellingen...")
    analysis_results = compute_polar_grid_slopes(
        points, PLANE_MODEL, R_STEP, THETA_STEP, 
        DIST_THRESH, 10, PLANE_THICKNESS, PLANE_ITERATIONS
    )
    visualize_polar_curvature(analysis_results, f"Timestamp: {TARGET_TIME}")
else:
    print("Kan frame niet laden.")

In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
from rosbags.highlevel import AnyReader
from pathlib import Path
from dateutil import parser
import struct

# --- 1. DATA LADEN UIT MCAP ---

def get_lidar_by_timestamp(bag_path, target_time_str, topic_name=None):
    """Zoekt het dichtstbijzijnde LiDAR frame op basis van een timestamp string."""
    bag_path_obj = Path(bag_path)
    
    # Zet de tijd-string om naar nanoseconden (Unix epoch)
    dt = parser.parse(target_time_str)
    target_ns = int(dt.timestamp() * 1e9)
    print(f"Zoeken naar tijdstip (NS): {target_ns}")

    with AnyReader([bag_path_obj]) as reader:
        # Automatisch topic selecteren als er geen is opgegeven
        if topic_name is None:
            connections = [x for x in reader.connections if 'PointCloud2' in x.msgtype]
            if not connections: 
                print("Geen PointCloud2 topics gevonden!")
                return None, None
            connection = connections[0]
            print(f"Geselecteerd topic: {connection.topic}")
        else:
            connections = [x for x in reader.connections if x.topic == topic_name]
            if not connections: 
                print(f"Topic {topic_name} niet gevonden!")
                return None, None
            connection = connections[0]

        try:
            # We gebruiken 'start' om direct naar de juiste tijd te springen
            for conn, timestamp, rawdata in reader.messages(connections=[connection], start=target_ns):
                msg = reader.deserialize(rawdata, conn.msgtype)
                
                # Punten extraheren uit bytes
                point_step = msg.point_step
                num_floats_per_point = point_step // 4
                raw_floats = np.frombuffer(msg.data, dtype=np.float32)
                
                # Voorkom reshape errors door alleen volledige punten te pakken
                total_complete_points = len(raw_floats) // num_floats_per_point
                clean_floats = raw_floats[:total_complete_points * num_floats_per_point]
                points = clean_floats.reshape(-1, num_floats_per_point)
                xyz = points[:, :3]

                # --- OPSCHONING (Cruciaal voor jouw foutmelding) ---
                # 1. Verwijder NaN en Inf waarden
                xyz = xyz[np.all(np.isfinite(xyz), axis=1)]
                
                # 2. Verwijder extreme uitschieters (>200m) om geheugenfouten te voorkomen
                dist_sq = np.sum(xyz**2, axis=1)
                xyz = xyz[dist_sq < 200**2]

                if len(xyz) == 0:
                    print("Waarschuwing: Geen geldige punten overgebleven na filtering.")
                    continue

                print(f"Gevonden frame op bag-tijd: {timestamp}")
                print(f"Aantal punten in frame: {len(xyz)}")
                return xyz, timestamp
        except Exception as e:
            print(f"Fout tijdens het lezen: {e}")
            
    return None, None

# --- 2. ANALYSE FUNCTIE ---

def compute_polar_grid_slopes(points, plane_model, r_step=2.0, theta_step_deg=10.0, 
                             distance_thresh=0.5, min_points=10, 
                             plane_thickness=0.05, plane_iterations=50):
    """Verdeelt de cloud in polaire cellen en berekent lokale hellingen."""
    a, b, c, d = plane_model
    normal = np.array([a, b, c]) / np.linalg.norm([a, b, c])

    # Grondpunten filteren
    distances = points @ normal + d
    ground_points = points[np.abs(distances) < distance_thresh]
    
    if len(ground_points) == 0:
        print("Geen grondpunten gevonden! Check PLANE_MODEL of DIST_THRESH.")
        return []

    x, y = ground_points[:, 0], ground_points[:, 1]
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)

    # Definieer bins
    r_max = min(r.max(), 100) # Beperk tot 100m voor visualisatie
    r_bins = np.arange(0, r_max + r_step, r_step)
    theta_bins = np.deg2rad(np.arange(-180, 180 + theta_step_deg, theta_step_deg))
    
    z_axis = np.array([0, 0, 1])
    results = []

    for i in range(len(r_bins) - 1):
        for j in range(len(theta_bins) - 1):
            mask = (r >= r_bins[i]) & (r < r_bins[i+1]) & \
                   (theta >= theta_bins[j]) & (theta < theta_bins[j+1])
            cell_pts = ground_points[mask]

            if len(cell_pts) < min_points: continue

            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(cell_pts)

            try:
                local_plane, _ = pcd.segment_plane(plane_thickness, 3, plane_iterations)
                l_norm = np.array(local_plane[:3])
                l_norm /= np.linalg.norm(l_norm)
                if l_norm[2] < 0: l_norm = -l_norm
                
                slope = np.degrees(np.arccos(np.clip(np.dot(l_norm, z_axis), -1, 1)))
                results.append({
                    'r_bounds': (r_bins[i], r_bins[i+1]),
                    'theta_bounds': (theta_bins[j], theta_bins[j+1]),
                    'slope': slope
                })
            except: continue
    return results

# --- 3. VISUALISATIE ---

def visualize_polar_curvature(results, title_text):
    if not results: 
        print("Niets om te visualiseren.")
        return
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='polar')
    
    slopes = np.array([res['slope'] for res in results])
    vmax = 15 # Schaal tot 15 graden
    cmap = plt.get_cmap('RdYlGn_r')

    for res in results:
        r0, r1 = res['r_bounds']
        t0, t1 = res['theta_bounds']
        ax.bar(t0, r1-r0, width=t1-t0, bottom=r0, 
               color=cmap(np.clip(res['slope']/vmax, 0, 1)), align='edge', alpha=0.8)

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, vmax))
    plt.colorbar(sm, ax=ax, label='Helling (graden)')
    plt.title(f"Polar Ground Curvature Map\n{title_text}")
    plt.show()

# --- 4. CONFIGURATIE EN UITVOERING ---

# Paden en Tijd
MCAP_FILE_PATH = r"C:\rosbag_0.mcap"
TARGET_TIME    = "2025-11-12 4:22:07.465 PM +01"

# Parameters
PLANE_MODEL      = (0, 0, 1, 1.8786) # (a, b, c, d)
R_STEP           = 2.0
THETA_STEP       = 10.0
DIST_THRESH      = 1.0  # Verhoogd voor robuustheid
PLANE_THICKNESS  = 0.05
PLANE_ITERATIONS = 50

# START
print("--- Start Proces ---")
points, found_ts = get_lidar_by_timestamp(MCAP_FILE_PATH, TARGET_TIME)

if points is not None:
    print("Berekenen van hellingen...")
    analysis_results = compute_polar_grid_slopes(
        points, PLANE_MODEL, R_STEP, THETA_STEP, 
        DIST_THRESH, 15, PLANE_THICKNESS, PLANE_ITERATIONS
    )
    visualize_polar_curvature(analysis_results, f"Timestamp: {TARGET_TIME}")
else:
    print("Kan frame niet laden.")